# Cross-View Geolocation — Training Notebook

Train satellite and phone encoders for cross-view geolocation using
the CV-Cities dataset and symmetric InfoNCE loss.

**Runtime:** Google Colab with T4 GPU

**Pipeline:**
1. Install dependencies & clone repo
2. Download CV-Cities dataset (subset)
3. Train ResNet-50 teacher encoders
4. Distill to MobileNetV3 student
5. Export to TFLite
6. Download trained models

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q torch torchvision datasets Pillow numpy mercantile requests tqdm

In [ ]:
# Mount Google Drive for persistence
from google.colab import drive
drive.mount('/content/drive')

# Create working directory
WORK_DIR = '/content/drive/MyDrive/geolocator'
!mkdir -p $WORK_DIR
!mkdir -p $WORK_DIR/checkpoints
!mkdir -p $WORK_DIR/data

## 2. Download CV-Cities Dataset

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile
import os

# Download selected cities (start with smallest)
cities_to_download = ['sydney', 'singapore', 'seattle']

for city in cities_to_download:
    print(f'Downloading {city}...')
    zip_path = hf_hub_download(
        repo_id='gaoshuang98/CV-Cities',
        filename=f'{city}.zip',
        repo_type='dataset',
    )
    print(f'  Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(f'{WORK_DIR}/data')
    print(f'  Done.')

# Download GPS dict
print('Downloading GPS metadata...')
gps_path = hf_hub_download(
    repo_id='gaoshuang98/CV-Cities',
    filename='gps_dict_10_cities.pkl',
    repo_type='dataset',
)
!cp $gps_path $WORK_DIR/data/

print('\nDataset ready.')
!ls -la $WORK_DIR/data/

## 3. Train Teacher Encoders (ResNet-50)

In [ ]:
import sys
sys.path.insert(0, '/content')  # Or wherever the repo is cloned

from src.training.train import train

# Train with the selected cities
sat_encoder, phone_encoder, history = train(
    data_dir=f'{WORK_DIR}/data',
    cities=cities_to_download,
    output_dir=f'{WORK_DIR}/checkpoints/teacher',
    epochs=50,
    batch_size=256,
    lr=3e-4,
    weight_decay=0.2,
    embed_dim=256,
    warmup_epochs=1,
    checkpoint_interval=10,
    device='cuda',
)

In [ ]:
# Plot training history
import matplotlib.pyplot as plt
import json

with open(f'{WORK_DIR}/checkpoints/teacher/history.json') as f:
    history = json.load(f)

epochs = [h['epoch'] for h in history]
losses = [h['loss'] for h in history]
r1 = [h.get('recall@1', 0) for h in history]
r5 = [h.get('recall@5', 0) for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')

ax2.plot(epochs, r1, label='R@1')
ax2.plot(epochs, r5, label='R@5')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Recall')
ax2.set_title('Validation Recall@K')
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Distill to MobileNetV3

In [ ]:
from src.training.distill import distill

student = distill(
    teacher_checkpoint=f'{WORK_DIR}/checkpoints/teacher/best_model.pt',
    data_dir=f'{WORK_DIR}/data',
    cities=cities_to_download,
    output_dir=f'{WORK_DIR}/checkpoints/student',
    epochs=30,
    batch_size=256,
    lr=1e-4,
    device='cuda',
)

## 5. Export to TFLite

In [ ]:
!pip install -q onnx onnxruntime tf2onnx

from src.training.export import export

export(
    checkpoint_path=f'{WORK_DIR}/checkpoints/teacher/best_model.pt',
    output_dir=f'{WORK_DIR}/exported',
    embed_dim=256,
    export_satellite=True,
    export_phone=True,
    quantize_tflite=True,
)

## 6. Download Models

In [ ]:
# List exported files
!ls -la $WORK_DIR/exported/

# The phone_encoder.tflite is what goes on the phone
# The satellite_encoder.onnx is used for desktop deployment
# Copy to Drive root for easy download
!cp $WORK_DIR/exported/phone_encoder.tflite /content/
!cp $WORK_DIR/exported/satellite_encoder.onnx /content/

print('Files ready for download:')
print('  phone_encoder.tflite - MobileNetV3 for on-device inference')
print('  satellite_encoder.onnx - ResNet-50 for desktop deployment')